## Set up and Imports

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve
from google.colab import drive
from sklearn.model_selection import train_test_split
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.covariance import EmpiricalCovariance
from transformers import DataCollatorWithPadding
import torch.nn.functional as F
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'
from IPython.display import display

In [ ]:
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/SNIPS_OOD_Project"
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

MODEL_NAME = "bert-base-uncased"
NUM_FOLDS = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Mounted at /content/drive


## Metryki ODD

In [ ]:
def compute_ood_metrics(logits, labels, ood_label_idx):
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    ood_scores = probs[:, ood_label_idx]
    y_true = (labels == ood_label_idx).astype(int)

    # AUROC
    auroc = roc_auc_score(y_true, ood_scores)
    # AUPR
    precision, recall, _ = precision_recall_curve(y_true, ood_scores)
    aupr = auc(recall, precision)
    # FPR95
    idx = np.argmin(np.abs(recall - 0.95))
    fpr95 = 1 - precision[idx]

    return auroc, aupr, fpr95

## SNIPS dataset

In [ ]:
print("--- Pobieranie zbioru SNIPS (7 klas) ---")
raw_ds = load_dataset("DeepPavlov/snips", "default")
train_df = pd.DataFrame(raw_ds['train'])
test_df = pd.DataFrame(raw_ds['test'])
snips_df = pd.concat([train_df, test_df], ignore_index=True) # Zawiera klasy 0-6

possible_text_cols = ['query', 'utterance', 'text', 'sentence']
for col in possible_text_cols:
    if col in snips_df.columns:
        snips_df = snips_df.rename(columns={col: 'text'})
        break

--- Pobieranie zbioru SNIPS (7 klas) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/366k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/43.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13084 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1400 [00:00<?, ? examples/s]

## Tokenizer and embeddings

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True,
        max_length=64   # (SNIPS nie potrzebuje 512)
    )

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def get_embeddings(model, dataset, batch_size=32):
    model.eval()
    model.to(device)

    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=data_collator
    )
    embs = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)

            embs.append(outputs.logits.detach().cpu().numpy())

    return np.concatenate(embs)

## Contrastive trainer

In [ ]:
class ContrastiveTrainer(Trainer):
    def __init__(self, *args, lambda_param=2.0, **kwargs):
        """
        lambda_param: Waga dla straty kontrastywnej (w artykule domyślnie używają 2.0)
        """
        super().__init__(*args, **kwargs)
        self.lambda_param = lambda_param

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs["output_hidden_states"] = True

        outputs = model(**inputs)

        loss_ce = outputs.loss

        hidden_states = outputs.hidden_states[-1]
        h = hidden_states[:, 0, :]

        labels = inputs.get("labels")

        batch_size = labels.size(0)
        d = h.size(1) # wymiar ukryty (dla BERTa 768)

        dist_matrix = torch.cdist(h, h, p=2) ** 2

        pos_mask = (labels.unsqueeze(0) == labels.unsqueeze(1))
        neg_mask = ~pos_mask

        pos_mask.fill_diagonal_(False)

        pos_counts = torch.clamp(pos_mask.sum(dim=1).float(), min=1.0)
        neg_counts = torch.clamp(neg_mask.sum(dim=1).float(), min=1.0)

        xi = (dist_matrix * pos_mask).max()

        l_pos = (dist_matrix * pos_mask).sum(dim=1) / pos_counts

        l_neg = (F.relu(xi - dist_matrix) * neg_mask).sum(dim=1) / neg_counts

        loss_margin = (l_pos.sum() + l_neg.sum()) / (d * batch_size)

        loss = loss_ce + self.lambda_param * loss_margin

        return (loss, outputs) if return_outputs else loss

## Cross-validation

In [ ]:
FOLDS_CONFIG = [
    {"ood": [5, 6], "name": "Kino"},
    {"ood": [3, 0], "name": "Muzyka"},
    {"ood": [2, 1], "name": "Usługi"},
    {"ood": [4, 5], "name": "Twórczość"},
    {"ood": [6, 1], "name": "Mieszany"}
]

## Główna pętla - MAHALANOBIS

In [ ]:
results_MAHALANOBIS = []

for fold_idx, config in enumerate(FOLDS_CONFIG):
    print(f"\n===== FOLD {fold_idx+1}: {config['name']} =====")

    ood_classes = config["ood"]
    id_classes = [c for c in range(7) if c not in ood_classes]

    id_df_full = snips_df[snips_df['label'].isin(id_classes)].copy()
    ood_df_full = snips_df[snips_df['label'].isin(ood_classes)].copy()

    train_id, test_id = train_test_split(
        id_df_full, test_size=0.2, stratify=id_df_full['label'], random_state=42
    )

    train_df_fold = train_id.copy()

    test_df_fold = pd.concat([test_id, ood_df_full]).sample(frac=1, random_state=42)

    mapping = {old_id: new_id for new_id, old_id in enumerate(id_classes)}

    train_df_fold["mapped_label"] = train_df_fold["label"].map(mapping)

    train_df_fold = train_df_fold.dropna(subset=["mapped_label"])
    train_df_fold["mapped_label"] = train_df_fold["mapped_label"].astype(int)

    train_ds = Dataset.from_pandas(
        train_df_fold[['text', 'mapped_label']].rename(columns={'mapped_label': 'label'})
    ).map(tokenize_fn, batched=True)

    test_ds = Dataset.from_pandas(test_df_fold).map(tokenize_fn, batched=True)

    train_ds = train_ds.remove_columns([col for col in train_ds.column_names if col not in ["input_ids", "attention_mask", "label"]])
    test_ds = test_ds.remove_columns([col for col in test_ds.column_names if col not in ["input_ids", "attention_mask"]])

    train_ds.set_format(type='torch')
    test_ds.set_format(type='torch')

    # model
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(id_classes)
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    training_args = TrainingArguments(
        output_dir="./tmp",
        num_train_epochs=3,
        per_device_train_batch_size=32,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
        fp16=True
    )

    trainer = ContrastiveTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        data_collator=data_collator,
        lambda_param=2.0 # Waga z artykułu
    )
    trainer.train()

    train_embeddings = get_embeddings(model, train_ds)
    test_embeddings = get_embeddings(model, test_ds)

    train_labels = train_df_fold['mapped_label'].values
    true_labels = test_df_fold['label'].values

    class_means = []
    for c in range(len(id_classes)):
        class_means.append(train_embeddings[train_labels == c].mean(axis=0))
    class_means = np.array(class_means)

    cov = EmpiricalCovariance().fit(train_embeddings)
    precision = cov.precision_

    def mahalanobis(x, mean):
        diff = x - mean
        return np.sqrt(diff @ precision @ diff.T)

    scores = []
    diff = test_embeddings[:, None, :] - class_means[None, :, :]
    scores = np.einsum('bij,jk,bik->bi', diff, precision, diff)
    scores = np.sqrt(scores).min(axis=1)

    scores = np.array(scores)

    y_true = np.array([1 if l in ood_classes else 0 for l in true_labels])

    auroc = roc_auc_score(y_true, scores)
    precision_arr, recall_arr, _ = precision_recall_curve(y_true, scores)
    aupr = auc(recall_arr, precision_arr)

    fpr, tpr, thresholds = roc_curve(y_true, scores)
    idx_95 = np.argmin(np.abs(tpr - 0.95))
    fpr95 = fpr[idx_95]

    print(f"AUROC: {auroc:.4f}")

    print(f"Generowanie wykresu t-SNE dla Foldu {fold_idx+1}...")

    X_fold = test_embeddings

    y_fold_names = []
    for label in true_labels:
        if label in ood_classes:
            y_fold_names.append(f"OOD (Klasa {label})")
        else:
            y_fold_names.append(f"ID (Klasa {label})")

    tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
    projections = tsne.fit_transform(X_fold)

    fig = px.scatter(
        x=projections[:, 0],
        y=projections[:, 1],
        color=y_fold_names,
        title=f"Przestrzeń ukryta BERT - Fold {fold_idx+1}: {config['name']} (Zb. Testowy)",
        labels={'color': 'Rodzaj intencji', 'x': 'Wymiar 1', 'y': 'Wymiar 2'},
        opacity=0.8
    )

    fig.update_traces(marker=dict(size=4))

    html_path = f"{SAVE_PATH}/mahalanobis_tsne_fold_{fold_idx+1}.html"
    fig.write_html(html_path)
    print(f"✅ Zapisano wykres: {html_path}")

    results_MAHALANOBIS.append({
        "fold": fold_idx+1,
        "scenario": config['name'],
        "auroc": auroc,
        "aupr": aupr,
        "fpr95": fpr95
    })


===== FOLD 1: Kino =====


Map:   0%|          | 0/8296 [00:00<?, ? examples/s]

Map:   0%|          | 0/6188 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Map:   0%|          | 0/6203 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.655077
200,0.112916
300,0.074827
400,0.045308
500,0.032567
600,0.016754
700,0.022907


AUROC: 0.8507
Generowanie wykresu t-SNE dla Foldu 5...
✅ Zapisano wykres: /content/drive/MyDrive/SNIPS_OOD_Project/mahalanobis_tsne_fold_5.html


## Leave One Out - Mahalanobis

In [ ]:
results_LOO_MAHALANOBIS = []

for fold_idx in range(7):
    print(f"\n===== FOLD {fold_idx+1} =====")

    ood_classes = [fold_idx]
    id_classes = [c for c in range(7) if c not in ood_classes]

    id_df_full = snips_df[snips_df['label'].isin(id_classes)].copy()
    ood_df_full = snips_df[snips_df['label'].isin(ood_classes)].copy()

    train_id, test_id = train_test_split(
        id_df_full, test_size=0.2, stratify=id_df_full['label'], random_state=42
    )

    train_df_fold = train_id.copy()

    test_df_fold = pd.concat([test_id, ood_df_full]).sample(frac=1, random_state=42)

    mapping = {old_id: new_id for new_id, old_id in enumerate(id_classes)}

    train_df_fold["mapped_label"] = train_df_fold["label"].map(mapping)

    train_df_fold = train_df_fold.dropna(subset=["mapped_label"])
    train_df_fold["mapped_label"] = train_df_fold["mapped_label"].astype(int)

    train_ds = Dataset.from_pandas(
        train_df_fold[['text', 'mapped_label']].rename(columns={'mapped_label': 'label'})
    ).map(tokenize_fn, batched=True)

    test_ds = Dataset.from_pandas(test_df_fold).map(tokenize_fn, batched=True)

    train_ds = train_ds.remove_columns([col for col in train_ds.column_names if col not in ["input_ids", "attention_mask", "label"]])
    test_ds = test_ds.remove_columns([col for col in test_ds.column_names if col not in ["input_ids", "attention_mask"]])

    train_ds.set_format(type='torch')
    test_ds.set_format(type='torch')

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(id_classes)
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    training_args = TrainingArguments(
        output_dir="./tmp",
        num_train_epochs=3,
        per_device_train_batch_size=32,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
        fp16=True
    )

    trainer = ContrastiveTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        data_collator=data_collator,
        lambda_param=2.0 # Waga z artykułu
    )
    trainer.train()

    train_embeddings = get_embeddings(model, train_ds)
    test_embeddings = get_embeddings(model, test_ds)

    train_labels = train_df_fold['mapped_label'].values
    true_labels = test_df_fold['label'].values

    class_means = []
    for c in range(len(id_classes)):
        class_means.append(train_embeddings[train_labels == c].mean(axis=0))
    class_means = np.array(class_means)

    cov = EmpiricalCovariance().fit(train_embeddings)
    precision = cov.precision_

    def mahalanobis(x, mean):
        diff = x - mean
        return np.sqrt(diff @ precision @ diff.T)

    scores = []
    diff = test_embeddings[:, None, :] - class_means[None, :, :]
    scores = np.einsum('bij,jk,bik->bi', diff, precision, diff)
    scores = np.sqrt(scores).min(axis=1)

    scores = np.array(scores)

    y_true = np.array([1 if l in ood_classes else 0 for l in true_labels])

    auroc = roc_auc_score(y_true, scores)
    precision_arr, recall_arr, _ = precision_recall_curve(y_true, scores)
    aupr = auc(recall_arr, precision_arr)

    fpr, tpr, thresholds = roc_curve(y_true, scores)
    idx_95 = np.argmin(np.abs(tpr - 0.95))
    fpr95 = fpr[idx_95]

    print(f"AUROC: {auroc:.4f}")

    print(f"Generowanie wykresu t-SNE dla Foldu {fold_idx+1}...")

    X_fold = test_embeddings

    y_fold_names = []
    for label in true_labels:
        if label in ood_classes:
            y_fold_names.append(f"OOD (Klasa {label})")
        else:
            y_fold_names.append(f"ID (Klasa {label})")

    tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
    projections = tsne.fit_transform(X_fold)

    fig = px.scatter(
        x=projections[:, 0],
        y=projections[:, 1],
        color=y_fold_names,
        title=f"Przestrzeń ukryta BERT - Fold {fold_idx+1} (Zb. Testowy)",
        labels={'color': 'Rodzaj intencji', 'x': 'Wymiar 1', 'y': 'Wymiar 2'},
        opacity=0.8
    )

    fig.update_traces(marker=dict(size=4))

    html_path = f"{SAVE_PATH}/mahalanobis_loo_tsne_fold_{fold_idx+1}.html"
    fig.write_html(html_path)
    print(f"✅ Zapisano wykres: {html_path}")

    results_LOO_MAHALANOBIS.append({
        "fold": fold_idx+1,
        "scenario": f"OOD_Class_{fold_idx+1}",
        "auroc": auroc,
        "aupr": aupr,
        "fpr95": fpr95
    })


===== FOLD 1 =====


Map:   0%|          | 0/9953 [00:00<?, ? examples/s]

Map:   0%|          | 0/4531 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map:   0%|          | 0/4544 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.709580
200,0.112186
300,0.093518
400,0.052340
500,0.044773
600,0.045948
700,0.027708
800,0.028084
900,0.022911


AUROC: 0.8015
Generowanie wykresu t-SNE dla Foldu 7...
✅ Zapisano wykres: /content/drive/MyDrive/SNIPS_OOD_Project/mahalanobis_loo_tsne_fold_7.html


## Główna pętla - R-angle

In [ ]:
results_RANGLE = []

for fold_idx, config in enumerate(FOLDS_CONFIG):
    print(f"\n===== FOLD {fold_idx+1}: {config['name']} =====")

    ood_classes = config["ood"]
    id_classes = [c for c in range(7) if c not in ood_classes]

    id_df_full = snips_df[snips_df['label'].isin(id_classes)].copy()
    ood_df_full = snips_df[snips_df['label'].isin(ood_classes)].copy()

    train_id, test_id = train_test_split(
        id_df_full,
        test_size=0.2,
        stratify=id_df_full['label'],
        random_state=42
    )

    train_df_fold = train_id
    test_df_fold = pd.concat([test_id, ood_df_full]).sample(frac=1, random_state=42)

    mapping = {old_id: new_id for new_id, old_id in enumerate(id_classes)}

    train_df_fold['mapped_label'] = train_df_fold['label'].map(mapping)

    train_ds = Dataset.from_pandas(
        train_df_fold[['text', 'mapped_label']].rename(columns={'mapped_label': 'label'})
    ).map(tokenize_fn, batched=True)

    test_ds = Dataset.from_pandas(test_df_fold).map(tokenize_fn, batched=True)

    train_ds = train_ds.remove_columns([col for col in train_ds.column_names if col not in ["input_ids", "attention_mask", "label"]])
    test_ds = test_ds.remove_columns([col for col in test_ds.column_names if col not in ["input_ids", "attention_mask"]])

    train_ds.set_format(type='torch')
    test_ds.set_format(type='torch')

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(id_classes)
    )

    training_args = TrainingArguments(
        output_dir="./tmp",
        num_train_epochs=3,
        per_device_train_batch_size=32,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
        fp16=True
    )

    trainer = ContrastiveTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        data_collator=data_collator,
        lambda_param=2.0 # Waga z artykułu
    )
    trainer.train()

    train_embeddings = get_embeddings(model, train_ds)
    test_embeddings = get_embeddings(model, test_ds)

    train_labels = train_df_fold['mapped_label'].values
    true_labels = test_df_fold['label'].values

    class_means = []
    for c in range(len(id_classes)):
        class_means.append(train_embeddings[train_labels == c].mean(axis=0))

    class_means = np.array(class_means)

    def r_angle(x, mean):
        cos_sim = cosine_similarity([x], [mean])[0][0]
        cos_sim = np.clip(cos_sim, -1.0, 1.0)
        return np.arccos(cos_sim)

    scores = []

    for x in test_embeddings:
        angles = [r_angle(x, m) for m in class_means]
        scores.append(min(angles))  # najbliższa klasa

    scores = np.array(scores)

    y_true = np.array([1 if l in ood_classes else 0 for l in true_labels])

    auroc = roc_auc_score(y_true, scores)

    precision_arr, recall_arr, _ = precision_recall_curve(y_true, scores)
    aupr = auc(recall_arr, precision_arr)

    fpr, tpr, thresholds = roc_curve(y_true, scores)
    idx_95 = np.argmin(np.abs(tpr - 0.95))
    fpr95 = fpr[idx_95]

    print(f"AUROC: {auroc:.4f}")

    print(f"Generowanie wykresu t-SNE dla Foldu {fold_idx+1}...")

    X_fold = test_embeddings

    y_fold_names = []
    for label in true_labels:
        if label in ood_classes:
            y_fold_names.append(f"OOD (Klasa {label})")
        else:
            y_fold_names.append(f"ID (Klasa {label})")

    tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
    projections = tsne.fit_transform(X_fold)

    fig = px.scatter(
        x=projections[:, 0],
        y=projections[:, 1],
        color=y_fold_names,
        title=f"Przestrzeń ukryta BERT - Fold {fold_idx+1}: {config['name']} (Zb. Testowy)",
        labels={'color': 'Rodzaj intencji', 'x': 'Wymiar 1', 'y': 'Wymiar 2'},
        opacity=0.8
    )

    fig.update_traces(marker=dict(size=4))

    html_path = f"{SAVE_PATH}/rangle_tsne_fold_{fold_idx+1}.html"
    fig.write_html(html_path)
    print(f"✅ Zapisano wykres: {html_path}")

    results_RANGLE.append({
        "fold": fold_idx + 1,
        "scenario": config["name"],
        "auroc": auroc,
        "aupr": aupr,
        "fpr95": fpr95
    })


===== FOLD 1: Kino =====


Map:   0%|          | 0/8296 [00:00<?, ? examples/s]

Map:   0%|          | 0/6188 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map:   0%|          | 0/6203 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.655077
200,0.112916
300,0.074827
400,0.045308
500,0.032567
600,0.016754
700,0.022907


AUROC: 0.9038
Generowanie wykresu t-SNE dla Foldu 5...
✅ Zapisano wykres: /content/drive/MyDrive/SNIPS_OOD_Project/rangle_tsne_fold_5.html


## Leave One Out - R-angle

In [ ]:
results_LOO_RANGLE = []

for fold_idx in range(7):
    print(f"\n===== FOLD {fold_idx+1} =====")

    ood_classes = [fold_idx]
    id_classes = [c for c in range(7) if c not in ood_classes]

    id_df_full = snips_df[snips_df['label'].isin(id_classes)].copy()
    ood_df_full = snips_df[snips_df['label'].isin(ood_classes)].copy()

    train_id, test_id = train_test_split(
        id_df_full,
        test_size=0.2,
        stratify=id_df_full['label'],
        random_state=42
    )

    train_df_fold = train_id
    test_df_fold = pd.concat([test_id, ood_df_full]).sample(frac=1, random_state=42)

    mapping = {old_id: new_id for new_id, old_id in enumerate(id_classes)}

    train_df_fold['mapped_label'] = train_df_fold['label'].map(mapping)

    train_ds = Dataset.from_pandas(
        train_df_fold[['text', 'mapped_label']].rename(columns={'mapped_label': 'label'})
    ).map(tokenize_fn, batched=True)

    test_ds = Dataset.from_pandas(test_df_fold).map(tokenize_fn, batched=True)

    train_ds = train_ds.remove_columns([col for col in train_ds.column_names if col not in ["input_ids", "attention_mask", "label"]])
    test_ds = test_ds.remove_columns([col for col in test_ds.column_names if col not in ["input_ids", "attention_mask"]])

    train_ds.set_format(type='torch')
    test_ds.set_format(type='torch')

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(id_classes)
    )

    training_args = TrainingArguments(
        output_dir="./tmp",
        num_train_epochs=3,
        per_device_train_batch_size=32,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
        fp16=True
    )

    trainer = ContrastiveTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        data_collator=data_collator,
        lambda_param=2.0 # Waga z artykułu
    )
    trainer.train()

    train_embeddings = get_embeddings(model, train_ds)
    test_embeddings = get_embeddings(model, test_ds)

    train_labels = train_df_fold['mapped_label'].values
    true_labels = test_df_fold['label'].values

    class_means = []
    for c in range(len(id_classes)):
        class_means.append(train_embeddings[train_labels == c].mean(axis=0))

    class_means = np.array(class_means)

    def r_angle(x, mean):
        cos_sim = cosine_similarity([x], [mean])[0][0]
        cos_sim = np.clip(cos_sim, -1.0, 1.0)
        return np.arccos(cos_sim)

    scores = []

    for x in test_embeddings:
        angles = [r_angle(x, m) for m in class_means]
        scores.append(min(angles))  # najbliższa klasa

    scores = np.array(scores)

    y_true = np.array([1 if l in ood_classes else 0 for l in true_labels])

    auroc = roc_auc_score(y_true, scores)

    precision_arr, recall_arr, _ = precision_recall_curve(y_true, scores)
    aupr = auc(recall_arr, precision_arr)

    fpr, tpr, thresholds = roc_curve(y_true, scores)
    idx_95 = np.argmin(np.abs(tpr - 0.95))
    fpr95 = fpr[idx_95]

    print(f"AUROC: {auroc:.4f}")

    print(f"Generowanie wykresu t-SNE dla Foldu {fold_idx+1}...")

    X_fold = test_embeddings

    y_fold_names = []
    for label in true_labels:
        if label in ood_classes:
            y_fold_names.append(f"OOD (Klasa {label})")
        else:
            y_fold_names.append(f"ID (Klasa {label})")

    tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
    projections = tsne.fit_transform(X_fold)

    fig = px.scatter(
        x=projections[:, 0],
        y=projections[:, 1],
        color=y_fold_names,
        title=f"Przestrzeń ukryta BERT - Fold {fold_idx+1} (Zb. Testowy)",
        labels={'color': 'Rodzaj intencji', 'x': 'Wymiar 1', 'y': 'Wymiar 2'},
        opacity=0.8
    )

    fig.update_traces(marker=dict(size=4))

    html_path = f"{SAVE_PATH}/rangle_loo_tsne_fold_{fold_idx+1}.html"
    fig.write_html(html_path)
    print(f"✅ Zapisano wykres: {html_path}")

    results_LOO_RANGLE.append({
        "fold": fold_idx + 1,
        "scenario": f"OOD_Class_{fold_idx+1}",
        "auroc": auroc,
        "aupr": aupr,
        "fpr95": fpr95
    })


===== FOLD 1 =====


Map:   0%|          | 0/9953 [00:00<?, ? examples/s]

Map:   0%|          | 0/4531 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Map:   0%|          | 0/4544 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.709580
200,0.112186
300,0.093518
400,0.052340
500,0.044773
600,0.045948
700,0.027708
800,0.028084
900,0.022911


AUROC: 0.9163
Generowanie wykresu t-SNE dla Foldu 7...
✅ Zapisano wykres: /content/drive/MyDrive/SNIPS_OOD_Project/rangle_loo_tsne_fold_7.html


## RESULTS

In [ ]:
df_MAHALANOBIS = pd.DataFrame(results_MAHALANOBIS)

print("\n==== FINAL MAHALANOBIS ====")
print(df_MAHALANOBIS)

summary = {
    "Metric": ["AUROC", "AUPR", "FPR95"],
    "Mean": [
        df_MAHALANOBIS["auroc"].mean(),
        df_MAHALANOBIS["aupr"].mean(),
        df_MAHALANOBIS["fpr95"].mean()
    ],
    "Std": [
        df_MAHALANOBIS["auroc"].std(),
        df_MAHALANOBIS["aupr"].std(),
        df_MAHALANOBIS["fpr95"].std()
    ]
}

df_summary = pd.DataFrame(summary)
print(df_summary)


==== FINAL MAHALANOBIS ====
   fold   scenario     auroc      aupr     fpr95
0     1       Kino  0.920976  0.963432  0.492048
1     2     Muzyka  0.719803  0.825328  0.846303
2     3     Usługi  0.806145  0.891851  0.792535
3     4  Twórczość  0.843232  0.923711  0.711807
4     5   Mieszany  0.850747  0.920303  0.715113
  Metric      Mean       Std
0  AUROC  0.828181  0.073447
1   AUPR  0.904925  0.051281
2  FPR95  0.711561  0.135010


In [ ]:
detailed_path = f"{SAVE_PATH}/wyniki_mahalanobis.csv"
df_MAHALANOBIS.to_csv(detailed_path, index=False)
print(f"Zapisano wyniki szczegółowe: {detailed_path}")

df_summary = pd.DataFrame(summary)

summary_path = f"{SAVE_PATH}/podsumowanie_mahalanobis.csv"
df_summary.to_csv(summary_path, index=False)

print(f"Zapisano podsumowanie statystyczne: {summary_path}")

Zapisano wyniki szczegółowe: /content/drive/MyDrive/SNIPS_OOD_Project/wyniki_mahalanobis.csv
Zapisano podsumowanie statystyczne: /content/drive/MyDrive/SNIPS_OOD_Project/podsumowanie_mahalanobis.csv


In [ ]:
df_LOO_MAHALANOBIS = pd.DataFrame(results_LOO_MAHALANOBIS)

print("\n==== FINAL LOO MAHALANOBIS ====")
print(df_LOO_MAHALANOBIS)

loo_summary = {
    "Metric": ["AUROC", "AUPR", "FPR95"],
    "Mean": [
        df_LOO_MAHALANOBIS["auroc"].mean(),
        df_LOO_MAHALANOBIS["aupr"].mean(),
        df_LOO_MAHALANOBIS["fpr95"].mean()
    ],
    "Std": [
        df_LOO_MAHALANOBIS["auroc"].std(),
        df_LOO_MAHALANOBIS["aupr"].std(),
        df_LOO_MAHALANOBIS["fpr95"].std()
    ]
}

df_loo_summary = pd.DataFrame(loo_summary)
print(df_loo_summary)


==== FINAL LOO MAHALANOBIS ====
   fold     scenario     auroc      aupr     fpr95
0     1  OOD_Class_1  0.641618  0.583359  0.816794
1     2  OOD_Class_2  0.908192  0.907824  0.499396
2     3  OOD_Class_3  0.870053  0.808266  0.448930
3     4  OOD_Class_4  0.770628  0.720560  0.732338
4     5  OOD_Class_5  0.920007  0.911721  0.407884
5     6  OOD_Class_6  0.813081  0.825975  0.747788
6     7  OOD_Class_7  0.801463  0.795770  0.741247
  Metric      Mean       Std
0  AUROC  0.817863  0.095657
1   AUPR  0.793354  0.113874
2  FPR95  0.627768  0.168700


In [ ]:
detailed_path = f"{SAVE_PATH}/wyniki_loo_mahalanobis.csv"
df_LOO_MAHALANOBIS.to_csv(detailed_path, index=False)
print(f"Zapisano wyniki szczegółowe: {detailed_path}")

df_loo_summary = pd.DataFrame(loo_summary)

summary_path = f"{SAVE_PATH}/podsumowanie_loo_mahalanobis.csv"
df_loo_summary.to_csv(summary_path, index=False)

print(f"Zapisano podsumowanie statystyczne: {summary_path}")

Zapisano wyniki szczegółowe: /content/drive/MyDrive/SNIPS_OOD_Project/wyniki_loo_mahalanobis.csv
Zapisano podsumowanie statystyczne: /content/drive/MyDrive/SNIPS_OOD_Project/podsumowanie_loo_mahalanobis.csv


In [ ]:
df_RANGLE = pd.DataFrame(results_RANGLE)

print("\n==== FINAL R-ANGLE ====")
print(df_RANGLE)

summary_rangle = {
    "Metric": ["AUROC", "AUPR", "FPR95"],
    "Mean": [
        df_RANGLE["auroc"].mean(),
        df_RANGLE["aupr"].mean(),
        df_RANGLE["fpr95"].mean()
    ],
    "Std": [
        df_RANGLE["auroc"].std(),
        df_RANGLE["aupr"].std(),
        df_RANGLE["fpr95"].std()
    ]
}

df_summary_rangle = pd.DataFrame(summary_rangle)
print(df_summary_rangle)


==== FINAL R-ANGLE ====
   fold   scenario     auroc      aupr     fpr95
0     1       Kino  0.977389  0.989535  0.114217
1     2     Muzyka  0.778140  0.835797  0.643306
2     3     Usługi  0.844033  0.903027  0.590402
3     4  Twórczość  0.959304  0.978329  0.202892
4     5   Mieszany  0.903772  0.949367  0.511347
  Metric      Mean       Std
0  AUROC  0.892527  0.082497
1   AUPR  0.931211  0.062948
2  FPR95  0.412433  0.238537


In [ ]:
detailed_path = f"{SAVE_PATH}/wyniki_rangle.csv"
df_RANGLE.to_csv(detailed_path, index=False)
print(f"Zapisano wyniki szczegółowe: {detailed_path}")

summary_path = f"{SAVE_PATH}/podsumowanie_rangle.csv"
df_summary_rangle.to_csv(summary_path, index=False)

print(f"Zapisano podsumowanie statystyczne: {summary_path}")

Zapisano wyniki szczegółowe: /content/drive/MyDrive/SNIPS_OOD_Project/wyniki_rangle.csv
Zapisano podsumowanie statystyczne: /content/drive/MyDrive/SNIPS_OOD_Project/podsumowanie_rangle.csv


In [ ]:
df_LOO_RANGLE = pd.DataFrame(results_LOO_RANGLE)

print("\n==== FINAL LOO R-ANGLE ====")
print(df_LOO_RANGLE)

summary_loo_rangle = {
    "Metric": ["AUROC", "AUPR", "FPR95"],
    "Mean": [
        df_LOO_RANGLE["auroc"].mean(),
        df_LOO_RANGLE["aupr"].mean(),
        df_LOO_RANGLE["fpr95"].mean()
    ],
    "Std": [
        df_LOO_RANGLE["auroc"].std(),
        df_LOO_RANGLE["aupr"].std(),
        df_LOO_RANGLE["fpr95"].std()
    ]
}

df_summary_loo_rangle = pd.DataFrame(summary_loo_rangle)
print(df_summary_loo_rangle)


==== FINAL LOO R-ANGLE ====
   fold     scenario     auroc      aupr     fpr95
0     1  OOD_Class_1  0.794662  0.725550  0.577742
1     2  OOD_Class_2  0.975106  0.964112  0.071687
2     3  OOD_Class_3  0.975590  0.953675  0.052887
3     4  OOD_Class_4  0.936708  0.906077  0.244651
4     5  OOD_Class_5  0.982263  0.973334  0.053902
5     6  OOD_Class_6  0.969843  0.965340  0.146018
6     7  OOD_Class_7  0.916257  0.905812  0.424950
  Metric      Mean       Std
0  AUROC  0.935776  0.066769
1   AUPR  0.913414  0.087447
2  FPR95  0.224548  0.205521


In [ ]:
detailed_path = f"{SAVE_PATH}/wyniki_loo_rangle.csv"
df_LOO_RANGLE.to_csv(detailed_path, index=False)
print(f"Zapisano wyniki szczegółowe: {detailed_path}")

summary_path = f"{SAVE_PATH}/podsumowanie_loo_rangle.csv"
df_summary_loo_rangle.to_csv(summary_path, index=False)

print(f"Zapisano podsumowanie statystyczne: {summary_path}")

Zapisano wyniki szczegółowe: /content/drive/MyDrive/SNIPS_OOD_Project/wyniki_loo_rangle.csv
Zapisano podsumowanie statystyczne: /content/drive/MyDrive/SNIPS_OOD_Project/podsumowanie_loo_rangle.csv
